In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import tensorflow as tf
from transformers import T5Tokenizer, TFT5ForConditionalGeneration

# 1. Load Dataset
df = pd.read_csv("../datasets/customer_support_large_dataset.csv")
# Build input/output text
df["input_text"] = "intent: " + df["category"] + " | query: " + df["query_text"]
df["target_text"] = df["response"]

# 2. Train/Val/Test Split
train_df, test_df = train_test_split(df, test_size=0.30, random_state=42)
val_df, test_df = train_test_split(test_df, test_size=0.50, random_state=42)

# 3. Tokenizer + Encoding
model_name = "t5-small"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = TFT5ForConditionalGeneration.from_pretrained(model_name, from_pt=True)

MAX_LEN = 128

def tokenize_function(examples):
    model_inputs = tokenizer(
        examples["input_text"].tolist(),
        max_length=MAX_LEN,
        padding="max_length",
        truncation=True,
        return_tensors="tf"
    )

    labels = tokenizer(
        examples["target_text"].tolist(),
        max_length=MAX_LEN,
        padding="max_length",
        truncation=True,
        return_tensors="tf"
    )["input_ids"]

    model_inputs["labels"] = labels
    return model_inputs

train_encodings = tokenize_function(train_df)
val_encodings = tokenize_function(val_df)
test_encodings = tokenize_function(test_df)

# 4. Build tf.data.Dataset
def make_tf_dataset(encodings, batch_size=16):
    dataset = tf.data.Dataset.from_tensor_slices((
        {
            "input_ids": encodings["input_ids"],
            "attention_mask": encodings["attention_mask"],
            "labels": encodings["labels"],
        }
    ))
    return dataset.shuffle(1000).batch(batch_size)

train_dataset = make_tf_dataset(train_encodings)
val_dataset = make_tf_dataset(val_encodings)
test_dataset = make_tf_dataset(test_encodings)

# 5. Compile Model
optimizer = tf.keras.optimizers.Adam(learning_rate=5e-5)

model.compile(optimizer=optimizer)

# 6. Train
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10   # you can increase if GPU/TPU is available
)

model.save_pretrained("./response_generation_model")
tokenizer.save_pretrained("./response_generation_model")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/242M [00:00<?, ?B/s]

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.
All PyTorch model weights were used when initializing TFT5ForConditionalGeneration.

All the weights of TFT5ForConditionalGeneration were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFT5ForConditionalGeneration for predictions without further training.
TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.


Epoch 1/10
219/219 [==============================] - ETA: 0s - loss: 1.5448

Training the model

In [ ]:
# 8. Demo Function
def generate_response(intent, query, max_length=50, temperature=0.7, top_p=0.9):
    input_text = f"intent: {intent} | query: {query}"
    inputs = tokenizer(input_text, return_tensors="tf", padding=True, truncation=True)

    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=max_length,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        top_k=50
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Example usage
print(generate_response("cancel_order", "Can you cancel this order?"))